**utilitaires :**

In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneA_50.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneB_50.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneB_100.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneA_100.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneB_75.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneA_25.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneA_75.h5
/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneB_25.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_8.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_6.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_2.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_5.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_4.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_3.h5
/kaggle/input/datasets/almonavarro/ah-trainingdata/scene_1.h5
/kag

In [2]:
H5_PATH = '/kaggle/input/datasets/almonavarro/ah-dataset-final/eval_sceneA_100.h5'
GT_CSV = '/kaggle/input/datasets/almonavarro/ground-truth-train-csv/ground_truth_train.csv'

In [3]:
import h5py
import numpy as np
import pandas as pd

def get_frame_id_from_prediction(h5_path, pred_ego_yaw, pred_ego_x=None, tolerance=1e-4):
    """
    Retrouve l'ID (index) de la frame correspondant à une ligne de prédiction.
    
    Arguments:
    - h5_path : Chemin vers le fichier de la scène (ex: 'scene_10.h5')
    - pred_ego_yaw : La valeur 'ego_yaw' de ta ligne de prédiction
    - pred_ego_x : (Optionnel) Ajoute une vérification sur X pour plus de sécurité
    """
    with h5py.File(h5_path, 'r') as f:
        dataset = f['lidar_points']
        
        # On extrait les limites des frames
        ego_yaw_all = dataset['ego_yaw'][:]
        changes = np.where(np.diff(ego_yaw_all) != 0)[0] + 1
        boundaries = [0] + changes.tolist()
        
        # On cherche la frame correspondante
        for frame_idx, start_idx in enumerate(boundaries):
            frame_yaw = ego_yaw_all[start_idx]
            
            # On utilise une tolérance car les flottants CSV vs H5 peuvent varier infimement
            if abs(frame_yaw - pred_ego_yaw) < tolerance:
                if pred_ego_x is not None:
                    frame_x = dataset['ego_x'][start_idx]
                    if abs(frame_x - pred_ego_x) > tolerance:
                        continue # Faux positif sur le yaw, on continue
                return frame_idx
                
    return -1 # Frame non trouvée

In [4]:
def count_objects_in_scene_frame(GT_CSV, h5_path, frame_index, tolerance=1e-2):
    """
    Compte et renvoie les objets du ground truth pour une scène et une frame données.
    
    Arguments:
    - GT_CSV : Chemin vers 'ground_truth_train.csv'
    - h5_path : Chemin vers le fichier de la scène (ex: 'scene_10.h5')
    - frame_index : Le numéro de la frame (0, 1, 2...)
    """
    # 1. Récupérer l'ego_pose de la frame demandée dans le H5
    with h5py.File(h5_path, 'r') as f:
        dataset = f['lidar_points']
        ego_yaw_all = dataset['ego_yaw'][:]
        changes = np.where(np.diff(ego_yaw_all) != 0)[0] + 1
        boundaries = [0] + changes.tolist()
        
        if frame_index >= len(boundaries):
            raise ValueError(f"La frame {frame_index} n'existe pas dans ce fichier.")
            
        start_idx = boundaries[frame_index]
        target_yaw = ego_yaw_all[start_idx]
        target_x = dataset['ego_x'][start_idx]
        target_y = dataset['ego_y'][start_idx]

    # 2. Chercher cette pose dans le Ground Truth CSV
    df_gt = pd.read_csv(GT_CSV)
    
    # On filtre avec une tolérance pour les imprécisions des flottants
    mask = (
        (abs(df_gt['ego_yaw'] - target_yaw) < tolerance) &
        (abs(df_gt['ego_x'] - target_x) < tolerance) &
        (abs(df_gt['ego_y'] - target_y) < tolerance)
    )
    
    objects_in_frame = df_gt[mask]
    
    return len(objects_in_frame), objects_in_frame

**Câble :**
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-

In [5]:
import h5py
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.decomposition import PCA
import os

# --- 1. PARAMÈTRES GLOBAUX ---
OUTPUT_CSV = '/kaggle/working/predictions_evalA100_cables.csv'

MIN_HEIGHT = 10.0              
MIN_SEGMENT_LENGTH = 1.5       
ANGLE_TOLERANCE = 10.0         
CORRIDOR_WIDTH = 25.0          
MIN_SPANNED_LENGTH = 50.0      
MIN_LINEAR_DENSITY = 0.5       

def angle_diff(a, b):
    diff = abs(a - b)
    return min(diff, 180 - diff)

def compute_gap_segmented_bboxes(points, gap_threshold=10.0):
    """
    Crée des Bounding Boxes le long du câble, coupées s'il y a un trou > 2m.
    Garde l'orientation globale, mais ajuste la Longueur ET la Largeur 
    pour coller exactement aux points de chaque segment !
    """
    if len(points) < 5:
        return []

    # 1. PCA GLOBALE : On calcule l'orientation sur TOUT le câble pour la stabilité
    xy_pts = points[:, :2]
    pca = PCA(n_components=2).fit(xy_pts)
    main_dir = pca.components_[0]  # Axe de la longueur (Orientation du câble)
    ortho_dir = pca.components_[1] # Axe de la largeur (Perpendiculaire)
    yaw = np.arctan2(main_dir[1], main_dir[0])

    # Projections de tous les points sur l'axe principal (pour trouver les trous)
    proj_main = np.dot(xy_pts, main_dir)

    # 2. TRI DES POINTS : On trie le long de l'axe principal
    sort_idx = np.argsort(proj_main)
    proj_main_sorted = proj_main[sort_idx]
    points_sorted = points[sort_idx]

    # 3. IDENTIFICATION DES COUPURES (Les "Trous" > 10m)
    segments = []
    current_segment = [0]

    for i in range(1, len(proj_main_sorted)):
        if proj_main_sorted[i] - proj_main_sorted[i-1] > gap_threshold:
            segments.append(current_segment)
            current_segment = [i]
        else:
            current_segment.append(i)
    segments.append(current_segment)

    # 4. CRÉATION DES SOUS-BOÎTES SUR-MESURE
    bboxes = []
    for seg_indices in segments:
        if len(seg_indices) < 3: 
            continue # On ignore les miettes isolées

        seg_pts = points_sorted[seg_indices]
        seg_xy = seg_pts[:, :2]
        
        # Projections LOCALES du segment sur les axes GLOBAUX
        seg_proj_main = np.dot(seg_xy, main_dir)
        seg_proj_ortho = np.dot(seg_xy, ortho_dir)

        # -- A. Longueur --
        seg_min_main = np.min(seg_proj_main)
        seg_max_main = np.max(seg_proj_main)
        length = max(seg_max_main - seg_min_main, 1.0) # Au moins 1m de long
        local_main_center = (seg_max_main + seg_min_main) / 2.0

        # -- B. Largeur (NOUVEAU: Ajustement Local Exact) --
        seg_min_ortho = np.min(seg_proj_ortho)
        seg_max_ortho = np.max(seg_proj_ortho)
        width = max(seg_max_ortho - seg_min_ortho, 0.3) # Au moins 30cm de large (très serré)
        local_ortho_center = (seg_max_ortho + seg_min_ortho) / 2.0

        # Centre X, Y reconstruit à partir des centres locaux
        center_xy = local_main_center * main_dir + local_ortho_center * ortho_dir
        center_x, center_y = center_xy[0], center_xy[1]

        # -- C. Hauteur robuste (Percentiles 5% - 95%) --
        z_vals = seg_pts[:, 2]
        z_min = np.percentile(z_vals, 5)
        z_max = np.percentile(z_vals, 100)
        height = max(z_max - z_min, 0.3) # Au moins 30cm de haut
        center_z = (z_max + z_min) / 2.0

        # Ajout de la boîte (Le yaw reste identique pour toutes les boîtes du câble !)
        bboxes.append((center_x, center_y, center_z, width, length, height, yaw))

    return bboxes

# --- 3. PIPELINE PRINCIPAL ---
print(f"🚀 Lancement de l'extraction sur : {os.path.basename(H5_PATH)}")
predictions = []

with h5py.File(H5_PATH, 'r') as f:
    dataset = f['lidar_points']
    ego_yaw_all = dataset['ego_yaw'][:]
    
    # Identification des frames
    changes = np.where(np.diff(ego_yaw_all) != 0)[0] + 1
    boundaries = [0] + changes.tolist() + [len(ego_yaw_all)]
    del ego_yaw_all
    
    total_frames = len(boundaries) - 1
    print(f"📊 {total_frames} frames détectées. Traitement en cours...")
    
    for n_index in range(total_frames):
        if n_index % 10 == 0:
            print(f"   ⏳ Traitement Frame {n_index}/{total_frames}...")
            
        # --- A. Extraction et Conversion ---
        frame_data = dataset[boundaries[n_index]:boundaries[n_index+1]]
        df_frame = pd.DataFrame(frame_data)
        df_frame = df_frame[df_frame['distance_cm'] > 0]
        
        if df_frame.empty: continue
            
        ego_pose = (df_frame['ego_x'].iloc[0], df_frame['ego_y'].iloc[0], 
                    df_frame['ego_z'].iloc[0], df_frame['ego_yaw'].iloc[0])
        
        azimuth = np.deg2rad(df_frame['azimuth_raw'] / 100.0)
        elevation = np.deg2rad(df_frame['elevation_raw'] / 100.0)
        dist = df_frame['distance_cm'] / 100.0

        x = dist * np.cos(elevation) * np.cos(azimuth)
        y = -dist * np.cos(elevation) * np.sin(azimuth) # Left-Handed
        z = dist * np.sin(elevation)
        points = np.column_stack((x, y, z))
        
        # --- B. Filtre du Sol ---
        df = pd.DataFrame(points, columns=['x', 'y', 'z'])
        df['gx'] = np.floor((df['x'] - df['x'].min()) / 5.0)
        df['gy'] = np.floor((df['y'] - df['y'].min()) / 5.0)
        valid_ground = df.groupby(['gx', 'gy'])['z'].agg(['min', 'count']).reset_index()
        valid_ground = valid_ground[valid_ground['count'] >= 5]
        
        if valid_ground.empty: continue
            
        tree_g = cKDTree(valid_ground[['gx', 'gy']].values)
        _, idx = tree_g.query(df[['gx', 'gy']].values)
        mask_high = (df['z'] - valid_ground['min'].values[idx]) > MIN_HEIGHT
        high_points = points[mask_high]
        
        if len(high_points) < 10: continue
            
        # --- C. Graphe KNN et Longueur ---
        tree = cKDTree(high_points[:, :2])
        distances, indices = tree.query(high_points[:, :2], k=3)
        
        segments_info = []
        angles = []
        for i in range(len(high_points)):
            p0 = high_points[i][:2]
            for j in [1, 2]:
                if distances[i, j] > MIN_SEGMENT_LENGTH:
                    idx_n = indices[i, j]
                    p_n = high_points[idx_n][:2]
                    angle = np.degrees(np.arctan2(p_n[1] - p0[1], p_n[0] - p0[0])) % 180
                    segments_info.append((p0, p_n, angle, i, idx_n, distances[i, j]))
                    angles.append(angle)
                    
        # --- D. Directions Majoritaires ---
        if not angles: continue
        counts, bins = np.histogram(angles, bins=36, range=(0, 180))
        sorted_bins = np.argsort(counts)[::-1]
        
        top_angles = []
        for b in sorted_bins:
            if counts[b] < 5: break
            ang = (bins[b] + bins[b+1]) / 2.0
            if all(angle_diff(ang, a) > 15 for a in top_angles):
                top_angles.append(ang)
                if len(top_angles) == 2: break
                    
        # --- E. Validation des Couloirs et Création des BBox ---
        for majority_angle in top_angles:
            aligned = [info for info in segments_info if angle_diff(info[2], majority_angle) <= ANGLE_TOLERANCE]
            if not aligned: continue
                
            rad_angle = np.radians(majority_angle)
            normal_vec = np.array([-np.sin(rad_angle), np.cos(rad_angle)])
            dir_vec = np.array([np.cos(rad_angle), np.sin(rad_angle)])
            
            projs_norm = np.array([np.dot((info[0]+info[1])/2.0, normal_vec) for info in aligned])
            
            # Meilleur couloir
            best_count, best_center = 0, 0
            for p in projs_norm:
                count = np.sum((projs_norm >= p - CORRIDOR_WIDTH/2) & (projs_norm <= p + CORRIDOR_WIDTH/2))
                if count > best_count:
                    best_count, best_center = count, p
                    
            corridor_segs = [aligned[i] for i, proj in enumerate(projs_norm) if abs(proj - best_center) <= CORRIDOR_WIDTH/2]
            if not corridor_segs: continue
                
            # Densité Linéaire
            proj_along = []
            total_len = 0.0
            cable_indices_3d = set()
            
            for info in corridor_segs:
                proj_along.extend([np.dot(info[0], dir_vec), np.dot(info[1], dir_vec)])
                total_len += info[5]
                cable_indices_3d.update([info[3], info[4]])

            # On calcule la longueur entre le 5ème et le 95ème centile pour ignorer les extrémités
            # robust_min = np.percentile(proj_along, 5)
            # robust_max = np.percentile(proj_along, 95)
            # spanned_length = robust_max - robust_min
            
            spanned_length = max(proj_along) - min(proj_along)
            
            # Validation Finale
            if spanned_length > MIN_SPANNED_LENGTH and (total_len / spanned_length) > MIN_LINEAR_DENSITY:
                cable_points = high_points[list(cable_indices_3d)]
                
                # NOUVEAU : Appel de la fonction de segmentation (Découpe tous les 15 mètres)
                bboxes = compute_gap_segmented_bboxes(cable_points, gap_threshold=10.0) 
                
                total_length = sum([bbox[4] for bbox in bboxes]) # index 4 = length
                if len(bboxes) <= 2 and total_length < 30.0:
                    continue
                    
                for bbox in bboxes:
                    cx, cy, cz, width, length, height, yaw = bbox
                    predictions.append({
                        'ego_x': ego_pose[0], 'ego_y': ego_pose[1], 'ego_z': ego_pose[2], 'ego_yaw': ego_pose[3],
                        'bbox_center_x': cx, 'bbox_center_y': cy, 'bbox_center_z': cz,
                        'bbox_width': width, 'bbox_length': length, 'bbox_height': height,
                        'bbox_yaw': yaw,
                        'class_ID': 1, 'class_label': 'Cable'
                    })
# --- 4. SAUVEGARDE FORMAT AIRBUS ---
df_preds = pd.DataFrame(predictions)
cols = ['ego_x', 'ego_y', 'ego_z', 'ego_yaw', 'bbox_center_x', 'bbox_center_y', 
        'bbox_center_z', 'bbox_width', 'bbox_length', 'bbox_height', 'bbox_yaw', 
        'class_ID', 'class_label']

if not df_preds.empty:
    df_preds = df_preds[cols]
    df_preds.to_csv(OUTPUT_CSV, index=False)
    print(f"\n🎉 SUCCÈS ! {len(df_preds)} câbles détectés dans toute la scène.")
    print(f"📁 Fichier sauvegardé : {OUTPUT_CSV}")
else:
    print("\n⚠️ Aucun câble n'a validé les critères stricts dans cette scène.")

🚀 Lancement de l'extraction sur : eval_sceneA_100.h5
📊 100 frames détectées. Traitement en cours...
   ⏳ Traitement Frame 0/100...
   ⏳ Traitement Frame 10/100...
   ⏳ Traitement Frame 20/100...
   ⏳ Traitement Frame 30/100...
   ⏳ Traitement Frame 40/100...
   ⏳ Traitement Frame 50/100...
   ⏳ Traitement Frame 60/100...
   ⏳ Traitement Frame 70/100...
   ⏳ Traitement Frame 80/100...
   ⏳ Traitement Frame 90/100...

🎉 SUCCÈS ! 71 câbles détectés dans toute la scène.
📁 Fichier sauvegardé : /kaggle/working/predictions_evalA100_cables.csv


**éolienne :**
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-

In [6]:
import h5py
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
import cv2
import os
from collections import defaultdict

# --- 1. PARAMÈTRES ---
OUTPUT_CSV = '/kaggle/working/predictions_evalA100_turbines.csv'

# Critères Pilier
GRID_RES = 0.5
MIN_PTS_IN_COLUMN = 50         
MIN_RADIUS = 1.0               
MAX_RADIUS = 4.5               
MIN_GAP_ANGLE = 160.0          
MAX_RADIAL_THICKNESS = 0.4     # 🚨 Réduit pour éliminer les faux positifs (arbres pleins)
MAX_Z_GAP = 2.5                
MIN_TOTAL_HEIGHT = 15.0        
NMS_DISTANCE = 10.0             

# Paramètres Voxel Growing (Moitié Haute)
BLADE_SEARCH_RADIUS = 60.0     
VOXEL_SIZE = 3.0               

# --- 2. PIPELINE PRINCIPAL ---
print(f"🚀 Génération CSV : Piliers ancrés + PCA Orientée sur {os.path.basename(H5_PATH)}")
predictions = []

with h5py.File(H5_PATH, 'r') as f:
    dataset = f['lidar_points']
    ego_yaw_all = dataset['ego_yaw'][:]
    changes = np.where(np.diff(ego_yaw_all) != 0)[0] + 1
    boundaries = [0] + changes.tolist() + [len(ego_yaw_all)]
    del ego_yaw_all
    
    total_frames = len(boundaries) - 1
    
    for n_index in range(total_frames):
        df_frame = pd.DataFrame(dataset[boundaries[n_index]:boundaries[n_index+1]])
        df_frame = df_frame[df_frame['distance_cm'] > 0]
        if df_frame.empty: continue
            
        ego_pose = (df_frame['ego_x'].iloc[0], df_frame['ego_y'].iloc[0], 
                    df_frame['ego_z'].iloc[0], df_frame['ego_yaw'].iloc[0])
        
        az = np.deg2rad(df_frame['azimuth_raw'] / 100.0)
        el = np.deg2rad(df_frame['elevation_raw'] / 100.0)
        dist = df_frame['distance_cm'] / 100.0
        points = np.column_stack((
            dist * np.cos(el) * np.cos(az),
            -dist * np.cos(el) * np.sin(az),
            dist * np.sin(el)
        ))
        
        gx = np.floor((points[:, 0] - np.min(points[:, 0])) / 10.0)
        gy = np.floor((points[:, 1] - np.min(points[:, 1])) / 10.0)
        df_g = pd.DataFrame({'gx': gx, 'gy': gy, 'z': points[:, 2]})
        ground_map = df_g.groupby(['gx', 'gy'])['z'].min().to_dict()
        
        z_ground_local = np.array([ground_map.get((xi, yi), 0) for xi, yi in zip(gx, gy)])
        high_pts = points[(points[:, 2] - z_ground_local) > 10.0]
        
        if len(high_pts) < 100: continue

        x_min, y_min = np.min(high_pts[:, 0]), np.min(high_pts[:, 1])
        px = ((high_pts[:, 0] - x_min) / GRID_RES).astype(int)
        py = ((high_pts[:, 1] - y_min) / GRID_RES).astype(int)
        
        heatmap, _, _ = np.histogram2d(px, py, bins=[np.max(px)+2, np.max(py)+2])
        img_dense = np.zeros_like(heatmap.T, dtype=np.uint8)
        img_dense[heatmap.T >= MIN_PTS_IN_COLUMN] = 255
        img_closed = cv2.morphologyEx(img_dense, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
        contours, _ = cv2.findContours(img_closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        frame_predictions = []

        for cnt in contours:
            (cx_px, cy_px), _ = cv2.minEnclosingCircle(cnt)
            cx = cx_px * GRID_RES + x_min
            cy = cy_px * GRID_RES + y_min
            
            dists_2d = np.linalg.norm(points[:, :2] - [cx, cy], axis=1)
            cyl_pts = points[dists_2d <= MAX_RADIUS + 1.0] 
            
            if len(cyl_pts) < 100: continue
                
            z_sorted = np.sort(cyl_pts[:, 2])
            z_gaps = np.diff(z_sorted)
            break_indices = np.where(z_gaps > MAX_Z_GAP)[0]
            
            seg_starts = np.insert(break_indices + 1, 0, 0)
            seg_ends = np.append(break_indices, len(z_sorted) - 1)
            
            max_h, best_z_min, best_z_max = 0, 0, 0
            for s_idx, e_idx in zip(seg_starts, seg_ends):
                h = z_sorted[e_idx] - z_sorted[s_idx]
                if h > max_h:
                    max_h, best_z_min, best_z_max = h, z_sorted[s_idx], z_sorted[e_idx]
                    
            if max_h < MIN_TOTAL_HEIGHT: continue
                
            z_mid = (best_z_max + best_z_min) / 2.0
            slice_mask = (cyl_pts[:, 2] >= z_mid - 2.5) & (cyl_pts[:, 2] <= z_mid + 2.5)
            slice_pts = cyl_pts[slice_mask]
            
            if len(slice_pts) < 30: continue
                
            pts_2d = np.array(slice_pts[:, :2], dtype=np.float32)
            (scx, scy), s_radius = cv2.minEnclosingCircle(pts_2d)
            
            # --- TESTS PILIER ---
            if MIN_RADIUS <= s_radius <= MAX_RADIUS:
                d_center = np.linalg.norm(pts_2d - [scx, scy], axis=1)
                if np.std(d_center) > MAX_RADIAL_THICKNESS or np.mean(d_center) < s_radius * 0.7:
                    continue 
                    
                angles = np.degrees(np.arctan2(pts_2d[:, 1] - scy, pts_2d[:, 0] - scx))
                angles = np.sort(angles % 360)
                max_gap = max(np.max(np.diff(angles)), 360.0 - (angles[-1] - angles[0]))
                
                if max_gap >= MIN_GAP_ANGLE:
                    # =========================================================
                    # ✅ PILIER VALIDÉ ! 
                    # =========================================================
                    
                    # 1. 🚨 FIX : Pousser le pilier vers le bas (Le point le plus bas de la colonne)
                    dist_to_pillar = np.linalg.norm(points[:, :2] - [scx, scy], axis=1)
                    pillar_col_pts = points[dist_to_pillar <= s_radius * 1.2]
                    if len(pillar_col_pts) > 0:
                        best_z_min = np.min(pillar_col_pts[:, 2]) # On ancre la boîte à la racine exacte !
                        best_z_max = np.max(pillar_col_pts[:, 2])

                    # 2. 🚨 FIX : Voxel Growing UNIQUEMENT sur la moitié haute du pilier
                    mid_z = (best_z_max + best_z_min) / 2.0
                    local_mask = (
                        (points[:, 0] >= scx - BLADE_SEARCH_RADIUS) & (points[:, 0] <= scx + BLADE_SEARCH_RADIUS) &
                        (points[:, 1] >= scy - BLADE_SEARCH_RADIUS) & (points[:, 1] <= scy + BLADE_SEARCH_RADIUS) &
                        (points[:, 2] >= mid_z) # <-- Bloque totalement l'aspiration du sol/buissons !
                    )
                    local_pts = points[local_mask]
                    
                    if len(local_pts) > 0:
                        voxels = np.floor(local_pts / VOXEL_SIZE).astype(int)
                        voxel_map = defaultdict(list)
                        for idx, v in enumerate(voxels):
                            voxel_map[tuple(v)].append(idx)
                            
                        hub_mask = (
                            (local_pts[:, 0] >= scx - 5.0) & (local_pts[:, 0] <= scx + 5.0) &
                            (local_pts[:, 1] >= scy - 5.0) & (local_pts[:, 1] <= scy + 5.0) &
                            (local_pts[:, 2] >= best_z_max - 5.0) & (local_pts[:, 2] <= best_z_max + 5.0)
                        )
                        seed_indices = np.where(hub_mask)[0]
                        visited_voxels = set(tuple(v) for v in voxels[seed_indices])
                        active_queue = list(visited_voxels)
                        
                        offsets = [(dx, dy, dz) for dx in [-1,0,1] for dy in [-1,0,1] for dz in [-1,0,1] if not (dx==0 and dy==0 and dz==0)]
                        occupied_voxels = set(voxel_map.keys())
                        
                        while active_queue:
                            cx_v, cy_v, cz_v = active_queue.pop(0)
                            for dx, dy, dz in offsets:
                                neighbor = (cx_v + dx, cy_v + dy, cz_v + dz)
                                if neighbor in occupied_voxels and neighbor not in visited_voxels:
                                    visited_voxels.add(neighbor)
                                    active_queue.append(neighbor)
                                    
                        turbine_indices = []
                        for v in visited_voxels:
                            turbine_indices.extend(voxel_map[v])
                            
                        # 3. 🚨 FIX : PCA pour l'Orientation et les dimensions
                        if len(turbine_indices) > 5:
                            turbine_pts = local_pts[turbine_indices]
                            xy_pts = turbine_pts[:, :2]
                            
                            pca = PCA(n_components=2).fit(xy_pts)
                            main_dir = pca.components_[0]  # Direction du rotor (Length)
                            ortho_dir = pca.components_[1] # Épaisseur de nacelle (Width)
                            byaw = np.arctan2(main_dir[1], main_dir[0])
                            
                            proj_main = np.dot(xy_pts, main_dir)
                            proj_ortho = np.dot(xy_pts, ortho_dir)
                            
                            # On s'assure que la boîte ne soit jamais plus fine que le pilier lui-même
                            bl = max(np.max(proj_main) - np.min(proj_main), s_radius * 2.0)
                            bw = max(np.max(proj_ortho) - np.min(proj_ortho), s_radius * 2.0)
                            
                            center_main = (np.max(proj_main) + np.min(proj_main)) / 2.0
                            center_ortho = (np.max(proj_ortho) + np.min(proj_ortho)) / 2.0
                            center_xy = center_main * main_dir + center_ortho * ortho_dir
                            
                            bx, by = center_xy[0], center_xy[1]
                            final_max_z = max(best_z_max, np.max(turbine_pts[:, 2]))
                            
                        else:
                            # Sécurité : Si aucune pale n'est trouvée (Pilier nu)
                            bx, by = scx, scy
                            bl, bw = s_radius * 2.0, s_radius * 2.0
                            byaw = 0.0
                            final_max_z = best_z_max

                    # 4. Construction de la Boîte
                    bh = final_max_z - best_z_min # Le plancher absolu !
                    bz = best_z_min + (bh / 2.0)
                    
                    frame_predictions.append({
                        'ego_x': ego_pose[0], 'ego_y': ego_pose[1], 'ego_z': ego_pose[2], 'ego_yaw': ego_pose[3],
                        'bbox_center_x': bx, 'bbox_center_y': by, 'bbox_center_z': bz,
                        'bbox_width': bw, 'bbox_length': bl, 'bbox_height': bh,
                        'bbox_yaw': byaw,
                        'pillar_cx': scx, 'pillar_cy': scy,
                        'class_ID': 3, 'class_label': 'Wind turbine'
                    })

        # 🚨 FILTRE NMS
        if frame_predictions:
            frame_predictions.sort(key=lambda x: x['bbox_width'])
            kept_predictions = []
            
            for current_box in frame_predictions:
                overlap = False
                for kept_box in kept_predictions:
                    dist = np.sqrt((current_box['pillar_cx'] - kept_box['pillar_cx'])**2 + 
                                   (current_box['pillar_cy'] - kept_box['pillar_cy'])**2)
                    if dist < NMS_DISTANCE:
                        overlap = True
                        break
                        
                if not overlap:
                    kept_predictions.append(current_box)
            
            for p in kept_predictions:
                p.pop('pillar_cx')
                p.pop('pillar_cy')
                predictions.append(p)

# --- 3. EXPORT CSV ---
df_preds = pd.DataFrame(predictions)

if not df_preds.empty:
    cols = ['ego_x', 'ego_y', 'ego_z', 'ego_yaw', 'bbox_center_x', 'bbox_center_y', 
            'bbox_center_z', 'bbox_width', 'bbox_length', 'bbox_height', 'bbox_yaw', 
            'class_ID', 'class_label']
    df_preds = df_preds[cols]
    df_preds.to_csv(OUTPUT_CSV, index=False)
    print(f"\n🎉 PERFECTION ATTEINTE ! {len(df_preds)} éoliennes orientées et ancrées sauvegardées.")
else:
    print("\n⚠️ Aucune éolienne validée.")

🚀 Génération CSV : Piliers ancrés + PCA Orientée sur eval_sceneA_100.h5

🎉 PERFECTION ATTEINTE ! 42 éoliennes orientées et ancrées sauvegardées.


**Finalisation :**
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-
-

In [12]:
import numpy as np
import pandas as pd
from scipy.ndimage import minimum_filter, maximum_filter
from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree
from scipy.sparse.csgraph import connected_components
import h5py
import os

# ==========================================
# 1. FONCTIONS DE BASE (Ton Collègue)
# ==========================================

def get_robust_ground_mask(points, cell_size=0.5, height_threshold=0.25):
    x_min, y_min = np.min(points[:, :2], axis=0)
    x_indices = ((points[:, 0] - x_min) / cell_size).astype(np.int32)
    y_indices = ((points[:, 1] - y_min) / cell_size).astype(np.int32)
    
    df = pd.DataFrame({'x': x_indices, 'y': y_indices, 'z': points[:, 2]})
    min_z_map = df.groupby(['x', 'y'])['z'].min().reset_index()
    
    grid_w, grid_h = x_indices.max() + 1, y_indices.max() + 1
    z_grid = np.full((grid_w, grid_h), np.nan)
    z_grid[min_z_map['x'], min_z_map['y']] = min_z_map['z']
    
    z_grid_filled = pd.DataFrame(z_grid).interpolate(axis=0).ffill().bfill().values
    
    ground_surface = minimum_filter(z_grid_filled, size=5)
    ground_surface = maximum_filter(ground_surface, size=5)
    
    ground_z_at_points = ground_surface[x_indices, y_indices]
    return (points[:, 2] - ground_z_at_points) > height_threshold


def get_3d_clusters_from_grid(points, mask_no_ground, grid_resolution=1.0, density_threshold=1):
    pts_no_ground = points[mask_no_ground]
    x, y = pts_no_ground[:, 0], pts_no_ground[:, 1]
    x_min, y_min = x.min(), y.min()
    
    x_idx = ((x - x_min) / grid_resolution).astype(np.int32)
    y_idx = ((y - y_min) / grid_resolution).astype(np.int32)
    
    df_pts = pd.DataFrame({'x_idx': x_idx, 'y_idx': y_idx, 'pt_idx': np.arange(len(x))})
    cell_counts = df_pts.groupby(['x_idx', 'y_idx']).size().reset_index(name='count')
    dense_cells = cell_counts[cell_counts['count'] > density_threshold].copy()
    
    if len(dense_cells) == 0:
        return np.full(len(x), -1)
    
    dense_cells['x_coord'] = dense_cells['x_idx'] * grid_resolution
    dense_cells['y_coord'] = dense_cells['y_idx'] * grid_resolution
    
    eps_distance = grid_resolution * 1.5 
    clustering = DBSCAN(eps=eps_distance, min_samples=1).fit(dense_cells[['x_coord', 'y_coord']])
    dense_cells['cluster_label'] = clustering.labels_
    
    df_pts = df_pts.merge(dense_cells[['x_idx', 'y_idx', 'cluster_label']], on=['x_idx', 'y_idx'], how='left')
    return df_pts['cluster_label'].fillna(-1).values.astype(np.int32)

# ==========================================
# 2. LOGIQUE MÉTIER (Ton Collègue)
# ==========================================

def process_lidar_frame(points, grid_resolution=1.0, density_threshold=1, min_height_meters=30, max_z_gap=3.0, merge_distance=20):
    mask_no_ground = get_robust_ground_mask(points)
    cluster_labels = get_3d_clusters_from_grid(points, mask_no_ground, grid_resolution, density_threshold)
    
    global_labels = np.full(len(points), -1)
    global_labels[mask_no_ground] = cluster_labels
    
    unique_clusters = set(global_labels)
    unique_clusters.discard(-1)
    
    next_new_label = max(unique_clusters) + 1 if unique_clusters else 0
    split_clusters = set()

    for cluster_id in list(unique_clusters):
        mask_cluster = (global_labels == cluster_id)
        cluster_points = points[mask_cluster]
        
        z_values = np.sort(cluster_points[:, 2])
        z_diffs = np.diff(z_values)
        
        if np.any(z_diffs > max_z_gap):
            gap_index = np.argmax(z_diffs)
            split_z_value = z_values[gap_index] + (z_diffs[gap_index] / 2.0)
            
            mask_top_part = mask_cluster & (points[:, 2] > split_z_value)
            
            global_labels[mask_top_part] = next_new_label
            split_clusters.add(cluster_id)      
            split_clusters.add(next_new_label)  
            next_new_label += 1
        else:
            split_clusters.add(cluster_id)

    filtered_clusters = set()
    for cluster_id in list(split_clusters):
        mask_cluster = (global_labels == cluster_id)
        cluster_points = points[mask_cluster]
        
        if len(cluster_points) == 0: continue
            
        cluster_height = cluster_points[:, 2].max() - cluster_points[:, 2].min()
        
        if cluster_height < min_height_meters:
            global_labels[mask_cluster] = -1 
        else:
            filtered_clusters.add(cluster_id)

    filtered_list = list(filtered_clusters)
    n_clusters = len(filtered_list)
    
    if n_clusters > 1:
        adj_matrix = np.zeros((n_clusters, n_clusters), dtype=bool)
        trees = {}
        for i, c_id in enumerate(filtered_list):
            trees[i] = cKDTree(points[global_labels == c_id][:, :3])
            
        for i in range(n_clusters):
            for j in range(i + 1, n_clusters):
                matches = trees[i].query_ball_tree(trees[j], r=merge_distance)
                if any(len(m) > 0 for m in matches):
                    adj_matrix[i, j] = True
                    adj_matrix[j, i] = True
                    
        n_components, component_labels = connected_components(adj_matrix, directed=False)
        
        new_global_labels = np.full_like(global_labels, -1)
        final_clusters = set()
        
        for i, old_c_id in enumerate(filtered_list):
            new_comp_id = component_labels[i] 
            new_global_labels[global_labels == old_c_id] = new_comp_id
            final_clusters.add(new_comp_id)
            
        global_labels = new_global_labels
    else:
        final_clusters = filtered_clusters

    return points, mask_no_ground, global_labels

# ==========================================
# 3. L'ARBITRE DE CLASSIFICATION SPATIALE
# ==========================================

def point_in_oriented_bbox(px, py, cx, cy, w, l, yaw, margin):
    dx, dy = px - cx, py - cy
    c, s = np.cos(-yaw), np.sin(-yaw)
    lx = dx * c - dy * s
    ly = dx * s + dy * c
    return (abs(lx) <= (l / 2.0 + margin)) and (abs(ly) <= (w / 2.0 + margin))

# -- Paramètres d'exécution --
CABLES_CSV = '/kaggle/working/predictions_evalA100_cables.csv'
TURBINES_CSV = '/kaggle/working/predictions_evalA100_turbines.csv'
scene_name = os.path.splitext(os.path.basename(H5_PATH))[0]  # "eval_sceneA_100"
OUTPUT_FINAL_CSV = f'/kaggle/working/predictions_{scene_name}_FINAL.csv'


MAX_LIDAR_DISTANCE = 250.0  
CORRIDOR_MARGIN = 15.0      
TURBINE_MATCH_DIST = 50.0   
ANTENNA_MAX_RATIO = 2.5     

print("🚀 Démarrage de la Fusion de Modèles...")

try:
    df_cables = pd.read_csv(CABLES_CSV)
    df_turbines = pd.read_csv(TURBINES_CSV)
    print(f"✅ Contextes chargés : {len(df_cables)} câbles, {len(df_turbines)} éoliennes.")
except FileNotFoundError:
    print("⚠️ Fichiers Câbles ou Éoliennes introuvables. Mode aveugle.")
    df_cables, df_turbines = pd.DataFrame(), pd.DataFrame()

# 🚨 Stockage exclusif des nouvelles prédictions
new_predictions = []

with h5py.File(H5_PATH, 'r') as f:
    dataset = f['lidar_points']
    ego_yaw_all = dataset['ego_yaw'][:]
    changes = np.where(np.diff(ego_yaw_all) != 0)[0] + 1
    boundaries = [0] + changes.tolist() + [len(ego_yaw_all)]
    del ego_yaw_all
    
    total_frames = len(boundaries) - 1
    
    for n_index in range(total_frames):
        if n_index % 5 == 0:
            print(f"⏳ Traitement de la frame {n_index}/{total_frames}...")
            
        df_frame = pd.DataFrame(dataset[boundaries[n_index]:boundaries[n_index+1]])
        df_frame = df_frame[df_frame['distance_cm'] > 0]
        df_frame = df_frame[(df_frame['distance_cm'] / 100.0) < MAX_LIDAR_DISTANCE] 
        
        if df_frame.empty: continue
            
        ego_pose = (df_frame['ego_x'].iloc[0], df_frame['ego_y'].iloc[0], 
                    df_frame['ego_z'].iloc[0], df_frame['ego_yaw'].iloc[0])
        
        az = np.deg2rad(df_frame['azimuth_raw'] / 100.0)
        el = np.deg2rad(df_frame['elevation_raw'] / 100.0)
        dist = df_frame['distance_cm'] / 100.0
        points = np.column_stack((
            dist * np.cos(el) * np.cos(az), -dist * np.cos(el) * np.sin(az), dist * np.sin(el)
        ))
        
        # 🚨 FIX : Masque strict au centimètre près sur X, Y et Yaw pour isoler LA bonne frame
        frame_cables = pd.DataFrame()
        frame_turbines = pd.DataFrame()
        
        if not df_cables.empty:
            mask_c = (abs(df_cables['ego_x'] - ego_pose[0]) < 1e-2) & \
                     (abs(df_cables['ego_y'] - ego_pose[1]) < 1e-2) & \
                     (abs(df_cables['ego_yaw'] - ego_pose[3]) < 1e-3)
            frame_cables = df_cables[mask_c]
            
        if not df_turbines.empty:
            mask_t = (abs(df_turbines['ego_x'] - ego_pose[0]) < 1e-2) & \
                     (abs(df_turbines['ego_y'] - ego_pose[1]) < 1e-2) & \
                     (abs(df_turbines['ego_yaw'] - ego_pose[3]) < 1e-3)
            frame_turbines = df_turbines[mask_t]

        # Appel du clustering du collègue (avec ses vrais paramètres pour tuer les faux arbres)
        _, _, labels = process_lidar_frame(
            points,
            grid_resolution=1.0, 
            density_threshold=1, 
            min_height_meters=30.0, 
            max_z_gap=3.0,
            merge_distance=20.0
        )
        
        unique_ids = set(labels) - {-1}
        
        for cid in unique_ids:
            cluster_pts = points[labels == cid]
            
            min_x, max_x = np.min(cluster_pts[:, 0]), np.max(cluster_pts[:, 0])
            min_y, max_y = np.min(cluster_pts[:, 1]), np.max(cluster_pts[:, 1])
            min_z, max_z = np.min(cluster_pts[:, 2]), np.max(cluster_pts[:, 2])
            
            cx, cy, cz = (max_x + min_x)/2.0, (max_y + min_y)/2.0, (max_z + min_z)/2.0
            bl, bw, bh = max(max_x - min_x, 1.0), max(max_y - min_y, 1.0), max_z - min_z
            
            # --- RÈGLE 1 : ÉOLIENNE DÉJÀ CONNUE ? ---
            is_known_turbine = False
            for _, t_row in frame_turbines.iterrows():
                if np.sqrt((cx - t_row['bbox_center_x'])**2 + (cy - t_row['bbox_center_y'])**2) < TURBINE_MATCH_DIST:
                    is_known_turbine = True
                    break
            if is_known_turbine: continue

            # --- RÈGLE 2 : PYLÔNE DANS CÂBLES ? ---
            is_pylon = False
            for _, c_row in frame_cables.iterrows():
                if point_in_oriented_bbox(cx, cy, c_row['bbox_width'], c_row['bbox_length'], 
                                          c_row['bbox_center_x'], c_row['bbox_center_y'], 
                                          c_row['bbox_yaw'], margin=CORRIDOR_MARGIN):
                    is_pylon = True
                    break
            
            if is_pylon:
                assigned_class, assigned_label = 2, 'Pole'
            else:
                # --- RÈGLE 3 : ANTENNE OU PALES ? ---
                ratio = max(bl / bw, bw / bl)
                if ratio <= ANTENNA_MAX_RATIO and bl < 20.0 and bw < 20.0:
                    assigned_class, assigned_label = 0, 'Antenna'
                else:
                    continue
                    
            # Ajout du nouvel objet uniquement
            new_predictions.append({
                'ego_x': ego_pose[0], 'ego_y': ego_pose[1], 'ego_z': ego_pose[2], 'ego_yaw': ego_pose[3],
                'bbox_center_x': cx, 'bbox_center_y': cy, 'bbox_center_z': cz,
                'bbox_width': bw, 'bbox_length': bl, 'bbox_height': bh,
                'bbox_yaw': 0.0,
                'class_ID': assigned_class, 'class_label': assigned_label
            })


# ==========================================
# 4. CONCATÉNATION FINALE & EXPORTATION
# ==========================================
df_new = pd.DataFrame(new_predictions)

# Concaténation pure de tes CSV originaux + Nouveaux objets
frames_to_concat = []
if not df_cables.empty: frames_to_concat.append(df_cables)
if not df_turbines.empty: frames_to_concat.append(df_turbines)
if not df_new.empty: frames_to_concat.append(df_new)

if frames_to_concat:
    df_final = pd.concat(frames_to_concat, ignore_index=True)
    
    # =========================================================
    # 🚨 LE FILTRE POST-TRAITEMENT : DISTANCE POINT-PAR-POINT
    # =========================================================
    cables = df_final[df_final['class_ID'] == 1]
    targets_idx = df_final[df_final['class_ID'].isin([0, 3])].index # Antennes et Éoliennes
    
    changed_count = 0
    for idx in targets_idx:
        row = df_final.loc[idx]
        px, py = row['bbox_center_x'], row['bbox_center_y']
        
        # On isole les câbles de LA bonne frame
        frame_cables = cables[abs(cables['ego_yaw'] - row['ego_yaw']) < 1e-3]
        if frame_cables.empty: continue
            
        # 1. On "voxelise" les boîtes de câbles en générant 1 point tous les 2 mètres
        cable_pts = []
        for _, c_row in frame_cables.iterrows():
            cx, cy = c_row['bbox_center_x'], c_row['bbox_center_y']
            bl, byaw = c_row['bbox_length'], c_row['bbox_yaw']
            c, s = np.cos(byaw), np.sin(byaw)
            
            # On marche le long de la ligne centrale du câble
            for step in np.arange(-bl/2.0, bl/2.0, 2.0):
                cable_pts.append([cx + step * c, cy + step * s])
                
        if not cable_pts: continue
        
        # 2. Recherche du point du câble le plus proche via KDTree (Zéro erreur trigonométrique !)
        tree = cKDTree(cable_pts)
        dist, _ = tree.query([px, py])
        
        # 3. Validation de la distance (Bord à Câble < 50m)
        target_radius = max(row['bbox_width'], row['bbox_length']) / 2.0
        if (dist - target_radius) < 50.0:
            df_final.at[idx, 'class_ID'] = 2
            df_final.at[idx, 'class_label'] = 'Pole'
            changed_count += 1
            
    if changed_count > 0:
        print(f"🔄 Règle Spatiale (KDTree Point-par-Point) : {changed_count} objets convertis en Pylônes !")
    # =========================================================
    
    cols = ['ego_x', 'ego_y', 'ego_z', 'ego_yaw', 'bbox_center_x', 'bbox_center_y', 
            'bbox_center_z', 'bbox_width', 'bbox_length', 'bbox_height', 'bbox_yaw', 
            'class_ID', 'class_label']
    df_final = df_final[cols]
    df_final.to_csv(OUTPUT_FINAL_CSV, index=False)
    
    print(f"\n🎉 FUSION TERMINÉE ! Fichier généré : {OUTPUT_FINAL_CSV}")
    print("\n📊 Bilan des détections (Toutes Frames) :")
    print(df_final['class_label'].value_counts())
else:
    print("\n⚠️ Aucune prédiction générée.")

🚀 Démarrage de la Fusion de Modèles...
✅ Contextes chargés : 71 câbles, 42 éoliennes.
⏳ Traitement de la frame 0/100...
⏳ Traitement de la frame 5/100...
⏳ Traitement de la frame 10/100...
⏳ Traitement de la frame 15/100...
⏳ Traitement de la frame 20/100...
⏳ Traitement de la frame 25/100...
⏳ Traitement de la frame 30/100...
⏳ Traitement de la frame 35/100...
⏳ Traitement de la frame 40/100...
⏳ Traitement de la frame 45/100...
⏳ Traitement de la frame 50/100...
⏳ Traitement de la frame 55/100...
⏳ Traitement de la frame 60/100...
⏳ Traitement de la frame 65/100...
⏳ Traitement de la frame 70/100...
⏳ Traitement de la frame 75/100...
⏳ Traitement de la frame 80/100...
⏳ Traitement de la frame 85/100...
⏳ Traitement de la frame 90/100...
⏳ Traitement de la frame 95/100...
🔄 Règle Spatiale (KDTree Point-par-Point) : 7 objets convertis en Pylônes !

🎉 FUSION TERMINÉE ! Fichier généré : /kaggle/working/predictions_eval_sceneA_100_FINAL.csv

📊 Bilan des détections (Toutes Frames) :
class_

In [13]:
import h5py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# --- 1. FONCTION DE SOUS-ÉCHANTILLONNAGE UNIFORME ---
def voxel_grid_downsample(points, voxel_size):
    if len(points) == 0: return np.array([])
    vox_indices = np.floor(points / voxel_size).astype(int)
    voxel_map = {}
    for i, idx_tuple in enumerate(zip(vox_indices[:,0], vox_indices[:,1], vox_indices[:,2])):
        if idx_tuple not in voxel_map:
            voxel_map[idx_tuple] = points[i]
    return np.array(list(voxel_map.values()))

# --- 2. PARAMÈTRES ---
scene_name = os.path.splitext(os.path.basename(H5_PATH))[0]
PRED_CSV_PATH = f'/kaggle/working/predictions_{scene_name}_FINAL.csv'

BG_VOXEL_SIZE = 5.0 

# Couleurs par classe
COLORS_PTS = {0: 'mediumorchid', 1: 'cyan', 2: 'darkorange', 3: 'blue'}
COLORS_PRED_BOX = {0: 'magenta', 1: 'cyan', 2: 'orange', 3: 'red'}
CLASS_NAMES = {0: 'Antenna', 1: 'Cable', 2: 'Pole', 3: 'Wind turbine'}

# --- 3. PRÉ-CHARGEMENT DES DONNÉES ---
print("Analyse de la scène et des Prédictions...")
with h5py.File(H5_PATH, 'r') as f:
    ego_yaw_all = f['lidar_points']['ego_yaw'][:]
    changes = np.where(np.diff(ego_yaw_all) != 0)[0] + 1
    boundaries = [0] + changes.tolist() + [len(ego_yaw_all)]

try:
    df_preds = pd.read_csv(PRED_CSV_PATH)
except FileNotFoundError:
    print(f"⚠️ Fichier de prédictions non trouvé : {PRED_CSV_PATH}")
    df_preds = pd.DataFrame()

del ego_yaw_all

total_frames = len(boundaries) - 1
available_frames = list(range(total_frames))

# --- 4. L'INTERFACE INTERACTIVE ---
print(f"✅ {len(available_frames)} frames prêtes à l'inspection (Prédictions uniquement) !")

current_idx = 0
out_plot = widgets.Output()
btn_prev = widgets.Button(description="⬅️ Précédent", button_style='info')
btn_next = widgets.Button(description="Suivant ➡️", button_style='info')
dropdown = widgets.Dropdown(options=available_frames, description='Frame N°:')

def render_frame(frame_index):
    with out_plot:
        clear_output(wait=True)
        
        with h5py.File(H5_PATH, 'r') as f:
            start_idx = boundaries[frame_index]
            end_idx = boundaries[frame_index+1]
            df_frame = pd.DataFrame(f['lidar_points'][start_idx:end_idx])
            df_frame = df_frame[df_frame['distance_cm'] > 0]
            
            if df_frame.empty:
                print(f"Frame {frame_index} vide.")
                return
                
            target_yaw = df_frame['ego_yaw'].iloc[0]
            target_x = df_frame['ego_x'].iloc[0]
            target_y = df_frame['ego_y'].iloc[0]
            
            az = np.deg2rad(df_frame['azimuth_raw'] / 100.0)
            el = np.deg2rad(df_frame['elevation_raw'] / 100.0)
            dist = df_frame['distance_cm'] / 100.0
            points = np.column_stack((
                dist * np.cos(el) * np.cos(az),
                -dist * np.cos(el) * np.sin(az),
                dist * np.sin(el)
            ))

        # --- Filtrage Strict Prédictions ---
        if not df_preds.empty:
            mask_pred = (abs(df_preds['ego_x'] - target_x) < 1e-2) & \
                        (abs(df_preds['ego_y'] - target_y) < 1e-2) & \
                        (abs(df_preds['ego_yaw'] - target_yaw) < 1e-3)
            frame_preds = df_preds[mask_pred]
        else:
            frame_preds = pd.DataFrame()

        # Identification des points dans les boîtes prédites
        point_classes = np.full(len(points), -1)
        
        for _, row in frame_preds.iterrows():
            cx, cy, cz = row['bbox_center_x'], row['bbox_center_y'], row['bbox_center_z']
            bw, bl, bh = row['bbox_width'], row['bbox_length'], row['bbox_height']
            byaw = row.get('bbox_yaw', 0.0)
            cid = int(row['class_ID'])
            
            pts_centered = points - np.array([cx, cy, cz])
            c, s = np.cos(-byaw), np.sin(-byaw)
            rot_inv = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
            pts_local = np.dot(pts_centered, rot_inv.T)
            
            in_box = (
                (pts_local[:, 0] >= -bl/2.0) & (pts_local[:, 0] <= bl/2.0) &
                (pts_local[:, 1] >= -bw/2.0) & (pts_local[:, 1] <= bw/2.0) &
                (pts_local[:, 2] >= -bh/2.0) & (pts_local[:, 2] <= bh/2.0)
            )
            point_classes[in_box] = cid

        # --- Construction de la Scène 3D ---
        fig = go.Figure()
        
        # Nuage de points (Fond)
        bg_mask = (point_classes == -1)
        bg_plot = voxel_grid_downsample(points[bg_mask], BG_VOXEL_SIZE) 
        if len(bg_plot) > 0:
            fig.add_trace(go.Scatter3d(x=bg_plot[:, 0], y=bg_plot[:, 1], z=bg_plot[:, 2],
                mode='markers', marker=dict(size=1.5, color='darkgrey', opacity=1.0), name='Fond'
            ))
            
        # Points colorés par classe prédite
        for cid in np.unique(point_classes):
            if cid == -1: continue
            cls_pts = points[point_classes == cid]
            color = COLORS_PTS.get(cid, 'blue')
            name = CLASS_NAMES.get(cid, f'Classe {cid}')
            fig.add_trace(go.Scatter3d(x=cls_pts[:, 0], y=cls_pts[:, 1], z=cls_pts[:, 2],
                mode='markers', marker=dict(size=3, color=color, opacity=1.0), name=f'Pts: {name}'
            ))

        # Fonction de dessin des boîtes (uniquement utilisée pour les prédictions ici)
        def draw_bboxes(df_boxes, color_dict, prefix):
            lines_trace = []
            added_legends = set() 
            
            for i, row in df_boxes.iterrows():
                cx, cy, cz = row['bbox_center_x'], row['bbox_center_y'], row['bbox_center_z']
                bw, bl, bh = row['bbox_width'], row['bbox_length'], row['bbox_height']
                byaw = row.get('bbox_yaw', 0.0)
                cid = int(row['class_ID'])
                
                box_color = color_dict.get(cid, 'black')
                class_name = CLASS_NAMES.get(cid, f'Cls {cid}')
                legend_name = f"{prefix} {class_name}"
                
                show_leg = legend_name not in added_legends
                added_legends.add(legend_name)
                
                dx, dy, dz = bl/2.0, bw/2.0, bh/2.0 
                c, s = np.cos(byaw), np.sin(byaw)
                rot_mat = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
                
                corners_local = np.array([
                    [dx, dy, dz], [dx, -dy, dz], [-dx, -dy, dz], [-dx, dy, dz],
                    [dx, dy, -dz], [dx, -dy, -dz], [-dx, -dy, -dz], [-dx, dy, -dz]
                ])
                corners = np.dot(corners_local, rot_mat.T) + np.array([cx, cy, cz])
                
                lines = [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4), (0,4), (1,5), (2,6), (3,7)]
                for j, (p1, p2) in enumerate(lines):
                    lines_trace.append(go.Scatter3d(
                        x=[corners[p1, 0], corners[p2, 0]], 
                        y=[corners[p1, 1], corners[p2, 1]], 
                        z=[corners[p1, 2], corners[p2, 2]],
                        mode='lines', line=dict(color=box_color, width=4), 
                        showlegend=(show_leg and j==0), 
                        name=legend_name
                    ))
            return lines_trace

        if not frame_preds.empty:
            pred_traces = draw_bboxes(frame_preds, COLORS_PRED_BOX, 'Pred:')
            for trace in pred_traces: fig.add_trace(trace)

        fig.update_layout(
            title=f"Frame {frame_index} | Objets Prédits: {len(frame_preds)}", 
            scene=dict(aspectmode='data', bgcolor='white'),
            margin=dict(l=0, r=0, b=0, t=40)
        )
        fig.show()

# --- 5. LOGIQUE DES BOUTONS ---
def on_next(b):
    global current_idx
    if current_idx < len(available_frames) - 1:
        current_idx += 1
        dropdown.value = available_frames[current_idx]
    
def on_prev(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
        dropdown.value = available_frames[current_idx]
    
def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        global current_idx
        current_idx = available_frames.index(change['new'])
        render_frame(change['new'])

btn_next.on_click(on_next)
btn_prev.on_click(on_prev)
dropdown.observe(on_dropdown_change, names='value')

display(widgets.HBox([btn_prev, dropdown, btn_next]))
display(out_plot)

render_frame(available_frames[0])

Analyse de la scène et des Prédictions...
✅ 100 frames prêtes à l'inspection (Prédictions uniquement) !


Output()